# Backtesting & Strategy Evaluation

Loads linear regression test predictions from `../data/predictions.csv` and evaluates a long/short trading strategy. Compares against buy-and-hold AAPL and measures Beta vs S&P 500 and Sharpe Ratio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

preds = pd.read_csv('../data/predictions.csv', index_col=0, parse_dates=True)
y_test = preds['y_test'].values
y_pred = preds['y_pred'].values
split_date = preds.index[0]
print(f'Test period: {preds.index[0].date()} → {preds.index[-1].date()} ({len(preds)} days)')

## Conclusions — XGBoost

---

### Results Summary

| Model | MSE | Test R² | Train R² |
|-------|-----|---------|---------|
| Linear Regression | 0.000145 | 0.3092 | 0.5256 |
| RF from scratch (grid search) | 0.000199 | 0.0575 | 0.6082 |
| sklearn RandomForest | 0.000192 | 0.0896 | 0.6824 |
| C++ XGBoost | 0.000181 | 0.1400 | 0.9361 |
| sklearn XGBoost | 0.000179 | 0.1519 | 0.9379 |

---

### What I Learned

- **XGBoost massively overfits on this dataset.** Train R² of 0.936 against Test R² of 0.152 is the largest train/test gap of any model tried. The sequential boosting loop is extremely good at fitting the training data but the small dataset size (~220 rows) means those patterns do not hold up on the test set.

- **The C++ from-scratch implementation is close to sklearn.** Test R² of 0.140 vs 0.152 — a 0.012 gap. Unlike Random Forest where the from-scratch version was much further behind, XGBoost from scratch came within 8% of sklearn's performance. This validates that the implementation is correct.

- **XGBoost did not beat linear regression.** Despite being a far more complex model, the best XGBoost result (Test R² 0.152) is well below linear regression (0.309). The same pattern observed with Random Forest repeats: on small datasets, tree-based models overfit and linear regression generalises better.

- **Gradient boosting learns the training data too well.** With 100 trees each correcting the previous residuals, the ensemble eventually memorises the training set almost perfectly. Regularisation (`lambda`, `gamma`) and a lower learning rate could help, but the fundamental constraint is the small sample size.

- **Linear regression remains the best model for this dataset.** Its low complexity, combined with meaningful features like Momentum_5days, makes it the most generalisable model. The lesson is that model selection should be driven by the data, not by which algorithm is most sophisticated.

---

## Backtesting

Backtesting answers the question: *if I had traded based on my model's predictions, how much money would I have made?*

It turns a regression problem (predicting returns) into a trading simulation by converting each prediction into a **signal** and measuring the result against reality.

---

### Step 1 — Generate Signals

Each predicted return is converted into a trading decision using `np.sign()`:

- Predicted return > 0 → **+1** (go long: buy the stock)
- Predicted return < 0 → **-1** (go short: bet against the stock)

Boolean True/False would not work here because `False × actual_return = 0`, meaning you would sit out on short days instead of profiting from correctly predicting a down day.

---

### Step 2 — Strategy Returns

**Strategy return** = signal × actual return each day.

- Correct long (+1 × positive return) → profit
- Correct short (−1 × negative return) → negative × negative = profit
- Wrong prediction → loss

---

### Step 3 — Equity Curve

Daily returns are converted to growth multipliers — a 2% gain = 1.02, a 1% loss = 0.99. `np.cumprod(1 + strategy_returns)` chains all multipliers so each day's value builds on the previous. Both the strategy and benchmark start at 1.0.

---

### Assumptions and Limitations

This backtest is optimistic — it assumes zero transaction costs, free shorting, and instant daily position switching. In real trading these frictions significantly reduce returns. The results are an upper bound, not a realistic estimate.

---

## Beta Against the S&P 500

**Beta** measures how much a strategy moves relative to the overall market. It answers: *are my returns coming from a real signal, or am I just riding the market?*

$$\beta = \frac{\text{Cov}(\text{strategy returns}, \text{market returns})}{\text{Var}(\text{market returns})}$$

Interpretation:
- **Beta = 1** → moves exactly with the market
- **Beta < 1** → less sensitive to market moves (lower market exposure)
- **Beta = 0** → completely uncorrelated with the market
- **Beta < 0** → moves opposite to the market

**Result: Beta = 0.38**

A beta of 0.38 means the strategy moves at only 38% of the S&P 500's daily swings. Most of the strategy's returns come from stock-specific signals (momentum, volatility) rather than simply riding the market up. The gap between a 40% strategy return and a 15% buy-and-hold return, combined with a low beta, suggests the model is generating genuine **alpha** — returns independent of market movement.

`np.cov()` returns a 2×2 matrix; `[0][1]` picks the off-diagonal element which is the covariance between the two series.

---

## Sharpe Ratio

The **Sharpe Ratio** measures return per unit of risk. A strategy that makes 40% but swings wildly every day is very different from one that makes 40% smoothly — Sharpe captures this distinction.

$$\text{Daily Sharpe} = \frac{\text{mean(strategy returns)}}{\text{std(strategy returns)}}$$

To compare across strategies with different time scales, it is annualized by multiplying by $\sqrt{252}$:

$$\text{Annualized Sharpe} = \text{Daily Sharpe} \times \sqrt{252}$$

The $\sqrt{252}$ comes from how volatility scales with time. Variance scales linearly (× 252), so standard deviation scales with the square root (× √252). Mean scales fully (× 252). Dividing mean by std gives √252 as the annualisation factor.

**Benchmarks:**
- < 0 → losing money
- 0 to 0.5 → acceptable
- 0.5 to 1.0 → good
- 1.0 to 2.0 → very good
- > 2.0 → exceptional (often too good to be true)

**Result: Daily Sharpe = 0.58 (good range)**

The annualized Sharpe of 9.2 is artificially inflated because the test set is only ~45 days. With only 45 observations, a good run with low volatility produces an extreme annualized number. The daily Sharpe of 0.58 is the more honest and interpretable figure. This is a strong argument for expanding to 5 years of data — a proper ~250 day test set would produce a meaningful annualized Sharpe.

In [ ]:
#what array operations gives me just teh sign of each prediction
#np.sign() to get the sign of hte number
#edge case is zero np.sign(0)

signals = np.sign(y_pred)

strategy_returns = signals * y_test

comp_returns = np.cumprod(1+strategy_returns)

#the buy and hold benchmark is simplier: I just hold the stock every day so my daily
#returns are exactly y_test. Apply the same cumprod to those 

benchmark = np.cumprod(1+y_test)

In [ ]:
#visualization 

plt.figure(figsize=(12,4))
plt.plot(comp_returns,label='My strategy', alpha=0.7)
plt.plot(benchmark, label='Benchmarks', alpha=0.7)
plt.xlabel('Days')
plt.ylabel('Returns')
plt.title('Back Testing')
plt.legend()
plt.show()


In [ ]:
sp500 = yf.Ticker('^GSPC')
sp_data = sp500.history(period='1y')
sp_data['Return'] = sp_data['Close'].pct_change()

# get the date range of the test set
start_date = preds.index[0]
end_date = preds.index[-1]

# filter S&P 500 returns to the same date range
sp_test = sp_data['Return'].loc[start_date:end_date].dropna()
n = min(len(strategy_returns), len(sp_test))
beta = np.cov(strategy_returns[:n], sp_test.values[:n])[0][1] / np.var(sp_test.values[:n])
print(f'Beta: {beta:.4f}')

In [ ]:
daily_sharpe = np.mean(strategy_returns)/np.std(strategy_returns)

annual_sharpe = daily_sharpe*np.sqrt(252)

print(f'Daily Sharpe Ratio: {daily_sharpe:.4f}')
print(f'Annual Sharpe Ratio: {annual_sharpe:.4f}')


## What Is Next — 5 Years of Data

The backtesting results are promising but limited by the small dataset. The test set covers only ~45 trading days, which is not enough to draw robust conclusions about strategy performance.

**The fix is straightforward:** change `period='1y'` to `period='5y'` in the `yf.Ticker.history()` call at the top of the notebook. This gives ~1,250 rows after feature engineering, compared to ~220 now.

**What should improve:**

- **Test set size** — from ~45 days to ~250 days, making all metrics (Sharpe, Beta, equity curve) statistically meaningful
- **Model generalisation** — the feature weights learned by linear regression will be based on a much more diverse set of market conditions (bull runs, corrections, sideways periods), making them more reliable
- **Annualized Sharpe** — currently 9.2 due to the tiny sample; with 250 test days it will reflect a realistic long-run figure
- **Tree models may close the gap** — Random Forest and XGBoost struggled partly because 175 training rows is too few for trees to find stable splits. With 1,000 training rows the story may change

**One thing to watch:** the same feature set and model architecture will be reused — the goal is to see how results change purely from having more data, not from changing the model.